# ДЗ-1: распознавание английской речи (CTC)
Включите GPU: **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Вставьте ссылку на ZIP/TAR-архив с папкой `audio/` и, при наличии, `transcriptions/`.
Затем выполните **Среда выполнения → Выполнить все**. Остальные ячейки менять не нужно.
Код и веса загружаются автоматически. Если эталонов нет, сохраняются только предсказания.


In [ ]:
DATASET_URL = "" #@param {type:"string"}
REPOSITORY_URL = "https://github.com/NS1973/DLA-ASR-2026.git"

# Frozen source/archive settings are supplied with the final release.
SOURCE_ARCHIVE_URL = "https://drive.google.com/file/d/1VMYIOB54X3j9xr2gFsqXM0xOXTNBtFqR/view"
SOURCE_ARCHIVE_SHA256 = "947880caa84b9739388ddde5dbd0d98fd49518c3a4cbb0b9ef8aefb08ea19f62"
RELEASE_MANIFEST_URL = "https://drive.google.com/file/d/1JqLLoKmo2mUbkoS5VcXn5ntOek0AkMQf/view"
RELEASE_MANIFEST_SHA256 = "c003c6ab131f26a1a58322b13ba966543aba58f244c0c7b561da37db5c49ab47"


## 1. Установка проекта
Устанавливаем зафиксированные исходники и ресурсы. Для архива проверяем SHA-256 перед распаковкой; при обычном доступе клонируем сдаваемую ветку.


In [ ]:
import os, subprocess, sys, json
"""Verify and atomically install an exported Git ZIP before project imports."""
import hashlib
from pathlib import Path, PurePosixPath
import shutil
import stat
import tempfile
import zipfile


def extract_verified_source_archive(archive, expected_sha256, destination):
    """Install flat Git-archive contents; preserve any existing project."""
    archive, destination = Path(archive), Path(destination)
    if destination.exists():
        raise FileExistsError(f'Project already exists: {destination}')
    with archive.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    if actual != expected_sha256:
        raise ValueError(f'Source SHA-256 mismatch: {actual}')
    with zipfile.ZipFile(archive) as source:
        members, names = source.infolist(), set()
        if sum(member.file_size for member in members) > 128 * 2**20:
            raise ValueError('Source archive exceeds 128 MiB uncompressed')
        for member in members:
            name = PurePosixPath(member.filename)
            if (not name.parts or name.is_absolute() or '..' in name.parts
                    or '\\' in member.filename or any(':' in part for part in name.parts)
                    or stat.S_ISLNK(member.external_attr >> 16) or str(name) in names):
                raise ValueError(f'Unsafe source archive member: {member.filename}')
            names.add(str(name))
        required = {'inference.py', 'train.py', 'requirements.txt', 'src/asr/runtime.py'}
        if not required <= names:
            raise ValueError('Source archive lacks required ASR project files')
        destination.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.asr-source-', dir=destination.parent) as temporary:
            staging = Path(temporary) / 'project'
            staging.mkdir()
            for member in members:
                target = staging / member.filename
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with source.open(member) as incoming, target.open('xb') as outgoing:
                        shutil.copyfileobj(incoming, outgoing)
            if not all((staging / name).is_file() for name in required):
                raise ValueError('Required ASR source entry is not a regular file')
            staging.replace(destination)
    return destination

PROJECT = Path('/content/DLA-ASR-2026')
if SOURCE_ARCHIVE_URL and not RELEASE_MANIFEST_URL:
    raise ValueError('Frozen source archive requires a matching release manifest')
if SOURCE_ARCHIVE_URL or RELEASE_MANIFEST_URL:
    subprocess.run([sys.executable,'-m','pip','install','-q','gdown==5.2.0'],check=True)
    import gdown
if not PROJECT.exists():
    if SOURCE_ARCHIVE_URL:
        PROJECT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.asr-download-', dir=PROJECT.parent) as temporary:
            archive = Path(temporary)/'source.zip'
            gdown.download(SOURCE_ARCHIVE_URL, str(archive), fuzzy=True, quiet=False)
            extract_verified_source_archive(archive, SOURCE_ARCHIVE_SHA256, PROJECT)
        print('Frozen source archive SHA-256 verified')
    else:
        subprocess.run(['git','clone','--branch','codex/hw1-asr-overnight-20261014',REPOSITORY_URL,str(PROJECT)],check=True)
if RELEASE_MANIFEST_URL:
    with tempfile.TemporaryDirectory(prefix='.asr-manifest-', dir=PROJECT.parent) as temporary:
        manifest = Path(temporary)/'resources.json'
        gdown.download(RELEASE_MANIFEST_URL, str(manifest), fuzzy=True, quiet=False)
        with manifest.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != RELEASE_MANIFEST_SHA256:
            raise ValueError('Release manifest SHA-256 mismatch')
        release = json.loads(manifest.read_text(encoding='utf-8'))
        if not release.get('resources') or release.get('model_sha256') != release['resources'][0].get('sha256'):
            raise ValueError('Release manifest/model identity mismatch')
        manifest.replace(PROJECT/'resources.json')
    print('Frozen release manifest SHA-256 verified')
os.chdir(PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import torch
print('PyTorch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Загрузка модели
Скрипт проверяет SHA-256 весов по манифесту сдачи.

In [ ]:
import json
subprocess.run([sys.executable,'scripts/download_resources.py'],check=True)
assert Path('artifacts/model.pt').is_file()
manifest = json.loads(Path('resources.json').read_text())
INFERENCE_OPTIONS = manifest.get('inference_overrides', ['decoder=greedy'])
print('Настройки сдаваемой модели:', INFERENCE_OPTIONS)


## 3. Пример использования
Небольшой настоящий пример показывает инференс и отдельный расчёт метрик.

In [ ]:
subprocess.run([sys.executable,'inference.py','checkpoint=artifacts/model.pt','custom_dir=artifacts/example','output_dir=predictions/example','workers=0',*INFERENCE_OPTIONS],check=True)
subprocess.run([sys.executable,'calc_metrics.py','--reference-dir','artifacts/example/transcriptions','--prediction-dir','predictions/example'],check=True)
from IPython.display import Audio, display
audio = next(Path('artifacts/example/audio').iterdir())
display(Audio(filename=str(audio)))
uid = audio.stem
print('Эталон:', Path(f'artifacts/example/transcriptions/{uid}.txt').read_text().strip())
print('Распознано:', Path(f'predictions/example/{uid}.txt').read_text().strip())


## 4. Ваш набор данных
Поддерживаются прямая ссылка на архив и публичная ссылка Google Drive. Архив может содержать одну внешнюю папку.

In [ ]:
import json
if not DATASET_URL.strip():
    raise ValueError('Вставьте ссылку на ваш набор в поле DATASET_URL первой ячейки.')
from tempfile import mkdtemp
Path('data').mkdir(exist_ok=True)
USER_WORKSPACE = Path(mkdtemp(prefix='user-',dir='data')).resolve()
USER_PREDICTIONS = USER_WORKSPACE / 'predictions'
subprocess.run([sys.executable,'scripts/prepare_custom_dataset.py','--url',DATASET_URL,'--output-dir',str(USER_WORKSPACE/'dataset')],check=True)
dataset = (USER_WORKSPACE/'dataset/dataset_path.txt').read_text(encoding='utf-8').strip()
subprocess.run([sys.executable,'inference.py','checkpoint=artifacts/model.pt',f'custom_dir={dataset}',f'output_dir={USER_PREDICTIONS}','workers=0',*INFERENCE_OPTIONS],check=True)
if (Path(dataset)/'transcriptions').is_dir():
    subprocess.run([sys.executable,'calc_metrics.py','--reference-dir',str(Path(dataset)/'transcriptions'),'--prediction-dir',str(USER_PREDICTIONS),'--output',str(USER_PREDICTIONS/'metrics.json')],check=True)
    print(json.dumps(json.loads((USER_PREDICTIONS/'metrics.json').read_text()), ensure_ascii=False, indent=2))
else:
    print('Эталонов нет: метрики не рассчитываются. Предсказания:', USER_PREDICTIONS)


## 5. Скачать результат

In [ ]:
import shutil
shutil.make_archive('/content/asr_predictions','zip',str(USER_PREDICTIONS))
print('Архив предсказаний: /content/asr_predictions.zip. Он также доступен в панели Файлы.')
from google.colab import files
files.download('/content/asr_predictions.zip')
